# 03 · O gabarito (`goldenset.csv`)

**Desafio 1 — BRACIS 2026 / Jusbrasil**

Uma linha por citação esperada, com span, tipo e classe. É o alvo do treino e a
régua da validação local — e é a base que exige mais cuidado, porque o arquivo
distribuído não bate com a documentação.

Este notebook cobre: integridade dos spans, a lacuna documentada, a distribuição
das classes, e a diferença de superfície entre os dois níveis.

In [ ]:
import sys, re, collections
from pathlib import Path

for base in [Path.cwd(), *Path.cwd().parents]:
    if (base / "src" / "eda_utils.py").exists():
        sys.path.insert(0, str(base / "src"))
        break

import pandas as pd
import matplotlib.pyplot as plt
import eda_utils as u

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("raiz do projeto:", u.raiz_projeto())

In [ ]:
gab = u.carregar_gabarito()
docs = u.carregar_documentos()
df = pd.DataFrame(gab).drop(columns=["trecho_bruto"])
print(f"{len(df)} linhas · {df.documento_id.nunique()} documentos")
df.head(8)

## 1. Integridade dos spans

`inicio`/`fim` são offsets em **codepoints Unicode**, com `fim` exclusivo. O campo
`trecho` é a cópia literal de `texto[inicio:fim]`, com as quebras de linha
escapadas como a sequência de dois caracteres `\n`.

In [ ]:
conferem = sum(1 for r in gab if docs[r["documento_id"]][r["inicio"]:r["fim"]] == r["trecho"])
print(f"trecho == texto[inicio:fim]: {conferem}/{len(gab)}")

com_quebra = sum(1 for r in gab if chr(10) in r["trecho"])
print(f"trechos com quebra de linha interna: {com_quebra} ({100*com_quebra/len(gab):.0f}%)")

sobrepostos = 0
for doc, grupo in df.groupby("documento_id"):
    faixas = sorted(zip(grupo.inicio, grupo.fim))
    sobrepostos += sum(1 for a, b in zip(faixas, faixas[1:]) if b[0] < a[1])
print("spans sobrepostos no mesmo documento:", sobrepostos)

Os 195 spans conferem exatamente. Dois detalhes que quebram pipeline se passarem
despercebidos:

1. **Um quarto dos trechos contém quebra de linha no meio.** Um extrator que
   trabalhe linha a linha perde 25% das citações. O reconhecimento tem que operar
   sobre o texto corrido.
2. **Os `.txt` usam LF puro.** Se você ler com `newline=''` num arquivo que tivesse
   CRLF, ou converter as quebras, todos os offsets a partir da primeira linha saem
   deslocados e o IoU despenca. `u.carregar_documentos()` valida isso na carga.

## 2. O gabarito foi revisado — e o `data/README.md` não acompanhou

Em **04/09/2026** a organização publicou uma versão atualizada do goldenset,
removendo citações da classe `incompleta` que geravam confusão sobre o que é, de
fato, uma citação: trechos que não apontam para nenhuma fonte específica e apenas
remetem de forma difusa à matéria ou à orientação de um tribunal — *"normas de
regência da matéria"*, *"jurisprudência pacífica desta Corte"*.

O arquivo distribuído hoje tem **195** linhas. O `data/README.md` ainda descreve a
versão anterior, de **225** (116 + 109). Não é linha faltando: é documentação
defasada. Vale reproduzir o diagnóstico aqui, porque a diferença entre as duas
leituras muda o alvo do extrator.

In [ ]:
DOCUMENTADO = {(1, "real"): 52, (1, "inventada"): 32, (1, "incompleta"): 32,
               (2, "real"): 44, (2, "inventada"): 32, (2, "incompleta"): 33}

obs = df.groupby(["nivel", "classificacao"]).size()
comp = pd.DataFrame({
    "documentado": pd.Series(DOCUMENTADO),
    "no arquivo": obs,
}).fillna(0).astype(int)
comp["diferença"] = comp["no arquivo"] - comp["documentado"]
print(comp.to_string())
print("\ntotal documentado:", comp.documentado.sum(), "| total no arquivo:", comp["no arquivo"].sum())

In [ ]:
# As linhas foram removidas do CSV: a numeração de citacao_id tem buracos.
lacunas = []
for doc, grupo in df.groupby("documento_id"):
    nums = sorted(int(re.sub(r"\D", "", c)) for c in grupo.citacao_id)
    faltando = [n for n in range(1, max(nums) + 1) if n not in nums]
    if faltando:
        lacunas.append({"documento_id": doc, "presentes": len(nums), "maior_id": max(nums),
                        "ausentes": ", ".join("g" + str(n) for n in faltando)})
lac = pd.DataFrame(lacunas)
print(f"documentos com lacuna: {len(lac)} · ids ausentes: {sum(len(l['ausentes'].split(',')) for l in lacunas)}")
lac

In [ ]:
# O que está nas posições dos buracos? Exemplo: gen_n1_005 não tem g1 nem g2.
t = docs["gen_n1_005"]
primeiro_span = df[df.documento_id == "gen_n1_005"].inicio.min()
print(repr(t[420:primeiro_span]))

In [ ]:
# As expressões difusas continuam no texto — só deixaram de ser anotadas.
difusas = [r"normas\s+de\s+reg[êe]ncia", r"jurisprud[êe]ncia\s+pac[íi]fica",
           r"precedentes\s+desta\s+Casa", r"verbete\s+sumular",
           r"entendi[rn]{1,2}[mn]ento\s+su[mn]ulado"]
for p in difusas:
    rx = re.compile(p, re.I)
    no_texto = sum(len(rx.findall(t)) for t in docs.values())
    no_gabarito = sum(1 for r in gab if rx.search(r["trecho"]))
    print(f"  {p:<34} texto: {no_texto:>2}  |  gabarito: {no_gabarito}")

As 25 lacunas internas mais as que ficariam no fim de cada sequência fecham a
diferença de 30, e o que está nessas posições é exatamente o que a organização
descreveu: *"Merece registro a **jurisprudência pacífica desta Corte**"*, *"repousa
nas **normas de regência da matéria**"*. As expressões seguem no texto dos
documentos — elas simplesmente **não são citações** para efeito do desafio.

**O critério oficial, na frase da organização:** existe uma decisão concreta por trás
da citação, mas os dados fornecidos não permitem chegar a um registro único na base
canônica. O que sobrou como `incompleta` sempre nomeia alguma fonte — tribunal,
relator, ano ou classe processual.

**Consequência para o extrator, e ela é o contrário do que a contagem do README
sugere:** extrair "jurisprudência pacífica desta Corte" é **falso positivo de
verdade**, não artefato de gabarito incompleto. O léxico de expressões difusas serve
como filtro negativo — para *não* extrair — e não como alvo.

## 3. Distribuição das classes

In [ ]:
cruz = pd.crosstab([df.nivel, df.tipo], df.classificacao)
print(cruz.to_string())

cruz.plot(kind="bar", stacked=True, color=["#DD8452", "#C44E52", "#4C72B0"])
plt.ylabel("citações"); plt.title("Classe por nível e tipo")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

A `jurisprudencia` domina (165 de 195), mas a `lei` não é resíduo: são 30 citações,
e por serem só 18 os registros normativos da base, ela é a parte **mais resolvível**
do problema — o notebook 04 mostra que dá para acertar 100% dela com uma tabela de
13 linhas.

Note também que a métrica é **macro-F1 entre as três classes**: a `inventada`, com
64 exemplos, pesa tanto quanto a `real`, com 96. Otimizar o agregado sem olhar por
classe leva ao lugar errado.

## 4. O formato do span carrega sinal

Antes de qualquer feature semântica, o tamanho do trecho já separa as classes.

In [ ]:
df["tam"] = df.fim - df.inicio
print(df.groupby("classificacao")["tam"].describe()[["count", "min", "50%", "max"]].round(0).to_string())

fig, ax = plt.subplots()
for classe, cor in [("real", "#4C72B0"), ("inventada", "#C44E52"), ("incompleta", "#DD8452")]:
    ax.hist(df.loc[df.classificacao == classe, "tam"], bins=18, alpha=0.6, label=classe, color=cor)
ax.set_xlabel("tamanho do span (caracteres)"); ax.set_ylabel("citações")
ax.set_title("Incompletas são prosa longa; inventadas são identificadores curtos")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
df["pos_relativa"] = [r.inicio / len(docs[r.documento_id]) for r in df.itertuples()]
df.boxplot(column="pos_relativa", by="classificacao", grid=False)
plt.suptitle(""); plt.title("Posição relativa da citação no documento")
plt.ylabel("offset / tamanho do documento"); plt.tight_layout(); plt.show()

print(df.groupby("classificacao")["pos_relativa"].median().round(2).to_string())

As `incompleta` têm **mediana de 58 caracteres e mínimo de 33** — são frases
("julgado do STF proferido em 2024 pela relatoria de Dias Toffoli"), não
identificadores. As `inventada` têm mediana 20 e mínimo 13: são identificadores
curtos e bem formados. Sozinho, o tamanho do span já é um classificador razoável
entre `incompleta` e o resto.

## 5. Nível 1 versus nível 2: a mesma citação, duas superfícies

In [ ]:
jur = df[df.tipo == "jurisprudencia"].copy()

def separador(t):
    if re.search(r"/[A-Z]{2}\b", t):      return "/UF"
    if re.search(r"-\s*[A-Z]{2}\b", t):   return "- UF"
    if re.search(r"\([A-Z]{2}\)", t):     return "(UF)"
    return "sem UF"

def formato_numero(t):
    m = re.search(r"\d[\d.\s]{4,}", t)
    if not m:                       return "sem número longo"
    s = m.group(0)
    if "." in s and " " in s:       return "pontuado + espaço espúrio"
    if "." in s:                    return "pontuado"
    return "sem pontuação"

jur["separador"] = jur.trecho.map(separador)
jur["formato"] = jur.trecho.map(formato_numero)
print(pd.crosstab(jur.separador, jur.nivel).to_string())
print()
print(pd.crosstab(jur.formato, jur.nivel).to_string())

In [ ]:
# Trechos do nível 2 em que uma LETRA ocupa o lugar de um dígito.
suspeitos = [r for r in gab
             if r["nivel"] == 2 and re.search(r"[0-9][A-Za-z]|[A-Za-z][0-9]", r["trecho"])
             and not re.search(r"n[ºo°]", r["trecho"][:6])]
for r in suspeitos:
    print(f'  [{r["classificacao"]:<10}] {re.sub(r"[ ]+", " ", r["trecho"])}')

A mudança de superfície é sistemática. No nível 1, 53 das 86 citações de
jurisprudência usam `/UF` e 44 trazem o número pontuado. No nível 2, o `/UF` cai para
4 — troca por `- UF` (21) e
`(UF)` (11) — e o número pontuado cai de 44 para 12, com 40 citações trazendo os
dígitos colados. Como o índice FTS quebra tokens em não-alfanuméricos, essa é a
diferença entre encontrar e não encontrar o registro.

E há a categoria que a documentação não descreve explicitamente: `5úmula 211`,
`21737l8`, `1.528.4S5`, `1.45g.779`, `170076O`. A garantia do desafio é que *um
dígito nunca é trocado por outro dígito* — mas **uma letra pode ocupar o lugar de um
dígito**. Isso vale para a sigla também (`5úmula`), o que significa que até o
roteador que decide "isto é uma súmula" precisa ser tolerante a OCR.

## 6. As citações de lei têm uma assinatura própria

Trinta citações de `lei`, e um padrão que salta aos olhos quando se lista todas.

In [ ]:
leis = df[df.tipo == "lei"][["nivel", "classificacao", "trecho", "id_canonico"]].copy()
leis["trecho"] = leis.trecho.map(lambda t: re.sub(r"\s+", " ", t))
leis.sort_values(["classificacao", "trecho"]).reset_index(drop=True)

In [ ]:
# Último artigo de cada estatuto citado no corpus (tabela feita à mão).
ULTIMO_ARTIGO = {
    "Lei nº 13.105/2015": 1072, "Código de Processo Civil": 1072, "CPC": 1072,
    "Código Civil": 2046, "CC": 2046,
    "Consolidação das Leis do Trabalho": 922, "CLT": 922,
    "Constituição Federal": 250, "Constituição da República": 250, "Constituição Fedcral": 250,
    "Código de Processo Penal": 811, "CPP": 811,
    "Código Penal Militar": 410, "CPM": 410,
    "Código de Defesa do Consumidor": 119, "CDC": 119,
    "Código Eleitoral": 383,
    "Lei Complementar nº 64/1990": 26,
    "Lei nº 9.504/1997": 107,
    "Lei nº 13.467/2017": 6,
}
ARTIGO = re.compile(r"art(?:igo)?\.?\s*([\d.]+)", re.I)

def prever_por_faixa(trecho):
    m = ARTIGO.search(re.sub(r"\s+", " ", trecho))
    if not m:
        return "incompleta"
    numero = int(m.group(1).replace(".", ""))
    for nome, ultimo in ULTIMO_ARTIGO.items():
        if nome.lower() in trecho.lower():
            return "inventada" if numero > ultimo else "real"
    return "estatuto não mapeado"

leis["previsto"] = leis.trecho.map(prever_por_faixa)
print(pd.crosstab(leis.previsto, leis.classificacao).to_string())
print("\nacertos:", int((leis.previsto == leis.classificacao).sum()), "de", len(leis))

**Trinta em trinta.** Toda citação de lei marcada como `inventada` aponta para um
artigo que não existe no estatuto invocado: `art. 1.134` de um código que termina no
1.072, `art. 158` do CDC que tem 119 artigos, `art. 60` de uma lei com 6 artigos,
`art. 261`, `279`, `290` e `303` de uma Constituição que para no 250. Nenhuma
`real` viola a faixa.

Não é sorte de amostra: é o método do gerador. Para inventar uma citação de lei sem
correr o risco de acertar um dispositivo verdadeiro por acidente, o caminho natural é
sortear um número acima do último artigo. Isso torna a classe `lei` **decidível por
regra**, sem modelo — desde que a tabela de estatutos cubra o que aparece no conjunto
final. Uso essa regra como verificação cruzada no notebook 04, onde o caminho
principal é resolver contra os 13 dispositivos da base.

## Impressões

**O gabarito é confiável; quem está defasado é o `data/README.md`.** As 30 linhas a
menos são a revisão de 04/09, não perda de dados — e a revisão define a fronteira do
problema com precisão: só é citação o que aponta para uma fonte concreta. Isso torna
a classe `incompleta` mais fácil, não mais difícil, porque some o subtipo que não
tinha nenhum gancho sintático confiável.

**A classe é legível na forma antes de ser legível no conteúdo.** Tamanho do span,
presença de número, presença de sigla de tribunal — esse punhado de features separa
`incompleta` das outras duas quase perfeitamente. A fronteira difícil é
`real` × `inventada`, e ela **não é decidível pelo texto**: as duas são
identificadores bem formados, indistinguíveis sem consultar a base. Isso define a
arquitetura: extrator e classificador de forma primeiro, resolução contra a base
depois, e a decisão `real`/`inventada` sai da cardinalidade do resultado — nunca de
um modelo de linguagem olhando só para o trecho.

**O nível 2 não é "o mesmo com ruído": é uma distribuição de superfície diferente.**
O separador de UF praticamente inverte (`/UF` some, `- UF` e `(UF)` aparecem) e o
número perde a pontuação em dois terços dos casos. Como o peso é 2× e o índice FTS é
sensível a pontuação, **a normalização é onde está a nota** — não o classificador.